In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

import pandas as pd

sys.path.append("..")

from predql.converter import SConverter, TConverter

In [11]:
from relbench.datasets import get_dataset, get_dataset_names
from relbench.tasks import get_task_names

get_dataset_names()

['rel-amazon',
 'rel-avito',
 'rel-event',
 'rel-f1',
 'rel-hm',
 'rel-stack',
 'rel-trial']

# F1 Database

In [179]:
from relbench.datasets import get_dataset

dataset_f1 = get_dataset(name="rel-f1", download=True)

db_f1 = dataset_f1.get_db()

timestamps_f1 = pd.date_range(start="1960-05-13", end="2005-01-01 ", freq="ME")

sconverter_f1 = SConverter(db_f1)
tconverter_f1 = TConverter(db_f1, timestamps_f1)

In [180]:
get_task_names("rel-f1")

['driver-position', 'driver-dnf', 'driver-top3']

## driver-dnf

Task Description: For each driver predict the if they will DNF (did not finish) a race in the next 1 month. 

In [181]:
# RelBench table

from relbench.tasks.f1 import DriverDNFTask 

task_table_rb = DriverDNFTask(dataset_f1).make_table(db=db_f1, timestamps=timestamps_f1)
print(task_table_rb)

Table(df=
           date  driverId  did_not_finish
0    1999-04-30        43               1
1    1999-04-30        24               1
2    1999-04-30        56               1
3    1999-05-31        54               1
4    1999-05-31        64               1
...         ...       ...             ...
9651 1969-06-30       277               1
9652 1969-07-31       360               1
9653 1969-07-31       288               0
9654 1969-08-31       327               1
9655 1969-08-31       359               1

[9656 rows x 3 columns],
  fkey_col_to_pkey_table={'driverId': 'drivers'},
  pkey_col=None,
  time_col=date)


In [182]:
# PredQL table

predql_query = """
    PREDICT MAX(results.statusId, 1, 31, DAYS) != 1
    FOR EACH drivers.driverId
    WHERE MAX(results.statusId, 1, 31, DAYS) IS NOT NULL;
"""

task_table_predql = tconverter_f1.convert(predql_query)
print(task_table_predql)

------PREDICT_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    CASE
        WHEN main.fk IS NOT NULL THEN TRUE
        ELSE FALSE
    END
 AS label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        for_each.timestamp,
    FROM
        drivers parent
    JOIN
        (------CONDITION_START------
    SELECT
        *
    FROM
        (------AGGREGATION_START------
        SELECT
            parent.driverId AS fk,
            MAX(aggr_tbl.statusId)
         AS comp_col,
            time.timestamp AS timestamp
        FROM
            drivers parent
        CROSS JOIN
            timestamp_df time
        LEFT JOIN
            results aggr_tbl
        ON
            aggr_tbl.driverId = parent.driverId
        AND
            aggr_tbl.date >= time.timestamp + INTERVAL '1 DAY'
        AND
            aggr_tbl.date < time.timestamp + INTERVAL '31 DAY'
        GROUP BY
            time.timestamp, parent.driverId
        ------AGGREGATION_END-

In [183]:
df_rb = task_table_rb.df
filtered_df_rb = df_rb[df_rb['driverId'] == 45]
sorted_df_rb = filtered_df_rb.sort_values(by='date')
print(sorted_df_rb)

df_predql = task_table_predql.df()
filtered_df_predql = df_predql[df_predql['fk'] == 45]
print(filtered_df_predql)

           date  driverId  did_not_finish
4863 2004-02-29        45               1
7883 2004-03-31        45               1
6737 2004-04-30        45               1
1946 2004-05-31        45               1
9106 2004-06-30        45               1
1283 2004-07-31        45               1
7292 2004-08-31        45               1
7828 2004-09-30        45               1
      fk  timestamp  label
9508  45 2004-02-29   True
9528  45 2004-03-31   True
9548  45 2004-04-30   True
9569  45 2004-05-31   True
9589  45 2004-06-30   True
9610  45 2004-07-31   True
9633  45 2004-08-31   True
9654  45 2004-09-30   True


In [184]:
df_rb = task_table_rb.df
filtered_df_rb = df_rb[df_rb['driverId'] == 40]
sorted_df_rb = filtered_df_rb.sort_values(by='date')
print(sorted_df_rb)

df_predql = task_table_predql.df()
filtered_df_predql = df_predql[df_predql['fk'] == 40]
print(filtered_df_predql)

           date  driverId  did_not_finish
7354 1999-02-28        40               1
6803 1999-03-31        40               1
4806 1999-05-31        40               1
6074 1999-06-30        40               1
8480 1999-07-31        40               1
8485 1999-08-31        40               1
7244 1999-09-30        40               1
8490 2000-02-29        40               1
602  2000-03-31        40               1
6084 2000-04-30        40               1
6085 2000-05-31        40               1
6090 2000-06-30        40               1
5481 2000-07-31        40               1
7843 2000-08-31        40               0
5520 2000-09-30        40               1
1902 2001-05-31        40               1
6097 2001-06-30        40               1
1282 2004-07-31        40               1
7293 2004-08-31        40               1
7829 2004-09-30        40               1
      fk  timestamp  label
8629  40 1999-02-28   True
8651  40 1999-03-31   True
8695  40 1999-05-31   True
8717  40 1

In [188]:
renamed_df_rb = df_rb.rename(columns={'driverId': 'fk',
                                      'date' : 'timestamp',
                                      'did_not_finish': 'label'})
df_rb = renamed_df_rb.sort_values(by=['timestamp', 'fk'])

df_predql['label'] = df_predql['label'].astype(int)

print(df_rb)
print(df_predql)

      timestamp   fk  label
7147 1960-05-31  288      1
2395 1960-05-31  346      1
8370 1960-05-31  355      0
5367 1960-05-31  359      1
1117 1960-05-31  363      1
...         ...  ...    ...
1880 2004-09-30   34      1
7829 2004-09-30   40      1
7899 2004-09-30   43      1
7828 2004-09-30   45      1
5473 2004-09-30   46      1

[9656 rows x 3 columns]
       fk  timestamp  label
0     288 1960-05-31      1
1     346 1960-05-31      1
2     355 1960-05-31      0
3     359 1960-05-31      1
4     363 1960-05-31      1
...   ...        ...    ...
9651   34 2004-09-30      1
9652   40 2004-09-30      1
9653   43 2004-09-30      1
9654   45 2004-09-30      1
9655   46 2004-09-30      1

[9656 rows x 3 columns]


In [189]:
merged = pd.merge(
    df_rb,       
    df_predql,   
    on=['fk', 'timestamp'], 
    how='outer',       
    suffixes=('_rb', '_predql'), 
    indicator=True    
)

print(f"Only in RelBench:\n {merged[merged['_merge'] == 'left_only']}")
print(f"Only in PredQL:\n {merged[merged['_merge'] == 'right_only']}")
print(f"In both:\n {merged[merged['_merge'] == 'both']}")

Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
In both:
       timestamp   fk  label_rb  label_predql _merge
0    2000-02-29    1         1             1   both
1    2000-03-31    1         1             1   both
2    2000-04-30    1         1             1   both
3    2000-05-31    1         1             1   both
4    2000-06-30    1         1             1   both
...         ...  ...       ...           ...    ...
9651 1960-08-31  544         1             1   both
9652 1960-08-31  545         1             1   both
9653 1960-08-31  546         1             1   both
9654 1960-08-31  547         1             1   both
9655 1960-10-31  548         1             1   both

[9656 rows x 5 columns]


## driver-top3

Task Description: For each driver predict if they will qualify in the top-3 for a race in the next 1 month. 

In [190]:
# RelBench table

from relbench.tasks.f1 import DriverTop3Task

task_table_rb = DriverTop3Task(dataset_f1).make_table(db=db_f1, timestamps=timestamps_f1)
print(task_table_rb)

Table(df=
           date  driverId  qualifying
0    2004-06-30        46           0
1    2004-06-30        13           1
2    2004-07-31        12           0
3    2004-07-31        20           0
4    2004-08-31        30           1
...         ...       ...         ...
1285 2003-07-31         1           0
1286 2003-08-31         1           0
1287 2003-09-30        20           0
1288 2004-02-29         1           0
1289 2004-05-31         3           0

[1290 rows x 3 columns],
  fkey_col_to_pkey_table={'driverId': 'drivers'},
  pkey_col=None,
  time_col=date)


In [196]:
# PredQL table

predql_query = """
    PREDICT MIN(qualifying.position, 1, 31, DAYS) <= 3
    FOR EACH drivers.driverId
    WHERE MIN(qualifying.position, 1, 31, DAYS) IS NOT NULL;
"""

task_table_predql = tconverter_f1.convert(predql_query)
print(task_table_predql)

------PREDICT_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    CASE
        WHEN main.fk IS NOT NULL THEN TRUE
        ELSE FALSE
    END
 AS label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        for_each.timestamp,
    FROM
        drivers parent
    JOIN
        (------CONDITION_START------
    SELECT
        *
    FROM
        (------AGGREGATION_START------
        SELECT
            parent.driverId AS fk,
            MIN(aggr_tbl.position)
         AS comp_col,
            time.timestamp AS timestamp
        FROM
            drivers parent
        CROSS JOIN
            timestamp_df time
        LEFT JOIN
            qualifying aggr_tbl
        ON
            aggr_tbl.driverId = parent.driverId
        AND
            aggr_tbl.date >= time.timestamp + INTERVAL '1 DAY'
        AND
            aggr_tbl.date < time.timestamp + INTERVAL '31 DAY'
        GROUP BY
            time.timestamp, parent.driverId
        ------AGGREGATION_E

In [197]:
df_rb = task_table_rb.df
filtered_df_rb = df_rb[df_rb['driverId'] == 45]
sorted_df_rb = filtered_df_rb.sort_values(by='date')
print(sorted_df_rb)

df_predql = task_table_predql.df()
filtered_df_predql = df_predql[df_predql['fk'] == 45]
print(filtered_df_predql)

           date  driverId  qualifying
1058 2004-02-29        45           0
1174 2004-03-31        45           0
530  2004-04-30        45           0
485  2004-05-31        45           0
356  2004-06-30        45           0
644  2004-07-31        45           0
734  2004-08-31        45           0
1139 2004-09-30        45           0
      fk  timestamp  label
1142  45 2004-02-29  False
1162  45 2004-03-31  False
1182  45 2004-04-30  False
1203  45 2004-05-31  False
1223  45 2004-06-30  False
1244  45 2004-07-31  False
1267  45 2004-08-31  False
1288  45 2004-09-30  False


In [198]:
df_rb = task_table_rb.df
filtered_df_rb = df_rb[df_rb['driverId'] == 40]
sorted_df_rb = filtered_df_rb.sort_values(by='date')
print(sorted_df_rb)

df_predql = task_table_predql.df()
filtered_df_predql = df_predql[df_predql['fk'] == 40]
print(filtered_df_predql)

           date  driverId  qualifying
778  1999-02-28        40           0
605  2000-02-29        40           0
1257 2000-03-31        40           0
262  2000-09-30        40           0
645  2004-07-31        40           0
732  2004-08-31        40           0
1140 2004-09-30        40           0
      fk  timestamp  label
775   40 1999-02-28  False
842   40 2000-02-29  False
864   40 2000-03-31  False
886   40 2000-09-30  False
1240  40 2004-07-31  False
1263  40 2004-08-31  False
1286  40 2004-09-30  False


In [199]:
renamed_df_rb = df_rb.rename(columns={'driverId': 'fk',
                                      'date' : 'timestamp',
                                      'qualifying': 'label'})
df_rb = renamed_df_rb.sort_values(by=['timestamp', 'fk'])

df_predql['label'] = df_predql['label'].astype(int)

print(df_rb)
print(df_predql)

      timestamp  fk  label
1068 1994-02-28  21      0
97   1994-02-28  29      1
9    1994-02-28  43      0
198  1994-02-28  48      0
98   1994-02-28  49      0
...         ...  ..    ...
422  2004-09-30  34      0
1140 2004-09-30  40      0
1138 2004-09-30  43      0
1139 2004-09-30  45      0
195  2004-09-30  46      0

[1290 rows x 3 columns]
      fk  timestamp  label
0     21 1994-02-28      0
1     29 1994-02-28      1
2     43 1994-02-28      0
3     48 1994-02-28      0
4     49 1994-02-28      0
...   ..        ...    ...
1285  34 2004-09-30      0
1286  40 2004-09-30      0
1287  43 2004-09-30      0
1288  45 2004-09-30      0
1289  46 2004-09-30      0

[1290 rows x 3 columns]


In [200]:
merged = pd.merge(
    df_rb,       
    df_predql,   
    on=['fk', 'timestamp'], 
    how='outer',       
    suffixes=('_rb', '_predql'), 
    indicator=True    
)

print(f"Only in RelBench:\n {merged[merged['_merge'] == 'left_only']}")
print(f"Only in PredQL:\n {merged[merged['_merge'] == 'right_only']}")
print(f"In both:\n {merged[merged['_merge'] == 'both']}")

Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
In both:
       timestamp   fk  label_rb  label_predql _merge
0    2000-02-29    1         0             0   both
1    2000-03-31    1         0             0   both
2    2000-09-30    1         0             0   both
3    2001-08-31    1         0             0   both
4    2002-02-28    1         0             0   both
...         ...  ...       ...           ...    ...
1285 1994-08-31  112         0             0   both
1286 1994-08-31  113         0             0   both
1287 1994-09-30  114         0             0   both
1288 1994-10-31  114         0             0   both
1289 1994-10-31  115         0             0   both

[1290 rows x 5 columns]


## Entity Regression Tasks

### driver-position

Task Description: Predict the average finishing position of each driver all races in the next 2 months. 

In [201]:
# RelBench table

from relbench.tasks.f1 import DriverPositionTask

task_table_rb = DriverPositionTask(dataset_f1).make_table(db=db_f1, timestamps=timestamps_f1)
print(task_table_rb)

Table(df=
            date  driverId   position
0     2004-08-31        34  10.333333
1     2004-08-31        10   5.000000
2     2004-08-31        29   5.500000
3     2004-09-30         7   4.000000
4     1999-04-30        13   8.800000
...          ...       ...        ...
11983 1969-05-31       261  14.500000
11984 1969-05-31       340  13.000000
11985 1969-06-30       359   3.333333
11986 1969-06-30       327   1.333333
11987 1969-07-31       342   8.000000

[11988 rows x 3 columns],
  fkey_col_to_pkey_table={'driverId': 'drivers'},
  pkey_col=None,
  time_col=date)


In [202]:
# PredQL table

predql_query = """
    PREDICT AVG(results.positionOrder, 1, 61, DAYS)
    FOR EACH drivers.driverId
    WHERE AVG(results.positionOrder, 1, 61, DAYS) IS NOT NULL;
"""

task_table_predql = tconverter_f1.convert(predql_query)
print(task_table_predql)

------PREDICT_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    main.comp_col
 AS label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        for_each.timestamp,
    FROM
        drivers parent
    JOIN
        (------CONDITION_START------
    SELECT
        *
    FROM
        (------AGGREGATION_START------
        SELECT
            parent.driverId AS fk,
            AVG(aggr_tbl.positionOrder)
         AS comp_col,
            time.timestamp AS timestamp
        FROM
            drivers parent
        CROSS JOIN
            timestamp_df time
        LEFT JOIN
            results aggr_tbl
        ON
            aggr_tbl.driverId = parent.driverId
        AND
            aggr_tbl.date >= time.timestamp + INTERVAL '1 DAY'
        AND
            aggr_tbl.date < time.timestamp + INTERVAL '61 DAY'
        GROUP BY
            time.timestamp, parent.driverId
        ------AGGREGATION_END------
    )
    WHERE
        comp_col IS NOT NULL
    -

In [203]:
print(task_table_predql.df().shape)

(11988, 3)


In [204]:
df_rb = task_table_rb.df
filtered_df_rb = df_rb[df_rb['driverId'] == 45]
sorted_df_rb = filtered_df_rb.sort_values(by='date')
print(sorted_df_rb)

df_predql = task_table_predql.df()
filtered_df_predql = df_predql[df_predql['fk'] == 45]
print(filtered_df_predql)

            date  driverId  position
5958  2004-01-31        45     15.50
11249 2004-02-29        45     16.75
7463  2004-03-31        45     16.60
1545  2004-04-30        45     16.20
8262  2004-05-31        45     17.00
48    2004-06-30        45     16.80
4405  2004-07-31        45     17.00
6681  2004-08-31        45     17.00
7393  2004-09-30        45     16.50
       fk  timestamp  label
11813  45 2004-01-31  15.50
11833  45 2004-02-29  16.75
11853  45 2004-03-31  16.60
11874  45 2004-04-30  16.20
11896  45 2004-05-31  17.00
11918  45 2004-06-30  16.80
11942  45 2004-07-31  17.00
11965  45 2004-08-31  17.00
11986  45 2004-09-30  16.50


In [205]:
df_rb = task_table_rb.df
filtered_df_rb = df_rb[df_rb['driverId'] == 40]
sorted_df_rb = filtered_df_rb.sort_values(by='date')
print(sorted_df_rb)

df_predql = task_table_predql.df()
filtered_df_predql = df_predql[df_predql['fk'] == 40]
print(filtered_df_predql)

            date  driverId   position
2488  1999-01-31        40   9.000000
6970  1999-02-28        40  15.500000
1777  1999-03-31        40  22.000000
8951  1999-04-30        40  14.000000
11179 1999-05-31        40  15.000000
9701  1999-06-30        40  15.800000
5155  1999-07-31        40  14.400000
5160  1999-08-31        40  14.666667
6690  1999-09-30        40  15.500000
6696  2000-01-31        40   7.500000
5165  2000-02-29        40  11.500000
5912  2000-03-31        40  13.000000
9713  2000-04-30        40  10.500000
9714  2000-05-31        40  14.200000
9719  2000-06-30        40  15.200000
10452 2000-07-31        40   9.500000
7415  2000-08-31        40   9.000000
10459 2000-09-30        40  12.000000
5178  2001-04-30        40   7.000000
8219  2001-05-31        40  13.500000
9728  2001-06-30        40  20.000000
50    2004-06-30        40  14.000000
4404  2004-07-31        40  14.500000
6682  2004-08-31        40  14.333333
7394  2004-09-30        40  13.000000
       fk  t

In [206]:
renamed_df_rb = df_rb.rename(columns={'driverId': 'fk',
                                      'date' : 'timestamp',
                                      'position': 'label'})
df_rb = renamed_df_rb.sort_values(by=['timestamp', 'fk'])

print(df_rb)
print(df_predql)

       timestamp   fk  label
2068  1960-05-31  288  11.50
2755  1960-05-31  340   2.00
8805  1960-05-31  346  13.75
8043  1960-05-31  355   1.00
11840 1960-05-31  359   6.25
...          ...  ...    ...
8193  2004-09-30   34  10.00
7394  2004-09-30   40  13.00
7479  2004-09-30   43  14.00
7393  2004-09-30   45  16.50
10442 2004-09-30   46  16.50

[11988 rows x 3 columns]
        fk  timestamp  label
0      288 1960-05-31  11.50
1      340 1960-05-31   2.00
2      346 1960-05-31  13.75
3      355 1960-05-31   1.00
4      359 1960-05-31   6.25
...    ...        ...    ...
11983   34 2004-09-30  10.00
11984   40 2004-09-30  13.00
11985   43 2004-09-30  14.00
11986   45 2004-09-30  16.50
11987   46 2004-09-30  16.50

[11988 rows x 3 columns]


In [207]:
merged = pd.merge(
    df_rb,       
    df_predql,   
    on=['fk', 'timestamp'], 
    how='outer',       
    suffixes=('_rb', '_predql'), 
    indicator=True    
)

print(f"Only in RelBench:\n {merged[merged['_merge'] == 'left_only']}")
print(f"Only in PredQL:\n {merged[merged['_merge'] == 'right_only']}")
print(f"In both:\n {merged[merged['_merge'] == 'both']}")

Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
In both:
        timestamp   fk   label_rb  label_predql _merge
0     2000-01-31    1  14.000000     14.000000   both
1     2000-02-29    1  16.250000     16.250000   both
2     2000-03-31    1  17.666667     17.666667   both
3     2000-04-30    1  14.666667     14.666667   both
4     2000-05-31    1  13.400000     13.400000   both
...          ...  ...        ...           ...    ...
11983 1960-08-31  546  16.000000     16.000000   both
11984 1960-07-31  547  17.000000     17.000000   both
11985 1960-08-31  547  17.000000     17.000000   both
11986 1960-09-30  548  13.000000     13.000000   both
11987 1960-10-31  548  13.000000     13.000000   both

[11988 rows x 5 columns]
